# GeoPython 2026 — Día 1b: Programación Orientada a Objetos

**Curso de Análisis Espacial con Python**  
Gabinete de Formación del CSIC — Estación Biológica de Doñana, Sevilla

---

> Este notebook es complementario al Día 1. Aquí aprendemos a crear nuestras propias clases,
> la base de la mayoría de los paquetes que usaremos en el curso (GeoPandas, Rasterio, Shapely…).

## Contenido

1. ¿Qué es una clase?
2. Nuestra primera clase: `Perro`
3. Métodos y atributos en detalle
4. Herencia: `Animal` y `Perro`
5. Más conceptos: `__repr__`, `__str__`
6. Métodos de clase y estáticos (`@classmethod`, `@staticmethod`)
7. Properties y `@property`
8. Un ejemplo geoespacial: clase `PuntoGPS`

---
## 1. ¿Qué es una clase?

Una **clase** es una plantilla para crear objetos. Define qué *datos* (atributos) y qué
*comportamientos* (métodos) tendrán los objetos que creemos a partir de ella.

Piénsalo como el plano de un edificio: el plano es la clase, cada edificio construido es
una **instancia**.

```
Clase  →  Instancia
Perro  →  miperro = Perro('Cop', 'Chucho', 3)
list   →  mi_lista = [1, 2, 3]
```

De hecho ya hemos usado clases todo el tiempo: `str`, `list`, `dict` son clases de Python.

In [ ]:
# list es una clase de Python — [1,2,3] es una instancia
mi_lista = [1, 2, 3]
print(type(mi_lista))       # <class 'list'>
print(type(mi_lista).__mro__)  # jerarquía de herencia

---
## 2. Nuestra primera clase: `Perro`

Vamos a modelar un perro. Necesitamos:

- **Atributos** (datos): nombre, raza, edad, horas de paseo
- **Métodos** (acciones): ladrar, registrar horas de paseo, decir cuánto falta para salir

### El método `__init__`

`__init__` es el **constructor**: se llama automáticamente cuando creamos una instancia.
`self` es la referencia al objeto que se está creando — siempre es el primer parámetro.

In [ ]:
from datetime import datetime

class Perro:
    """Clase para modelar un perro y su rutina de paseos."""

    def __init__(self, nombre, raza, edad):
        """
        Constructor.
        
        Args:
            nombre (str): nombre del perro
            raza   (str): raza
            edad   (int): edad en años
        """
        self.nombre = nombre
        self.raza = raza
        self.edad = edad
        self.horas_out = set()   # usamos set para evitar duplicados

        print(f'Hola, soy {nombre}, un {raza} de {edad} año(s)')

In [ ]:
# Crear una instancia
miperro = Perro('Cop', 'Chucho', 3)

In [ ]:
# Acceder a atributos
print(miperro.nombre)
print(miperro.raza)
print(miperro.edad)

---
## 3. Métodos en detalle

Los métodos son funciones definidas dentro de la clase.  
Siempre reciben `self` como primer argumento, que les da acceso a los atributos del objeto.

In [ ]:
from datetime import datetime

class Perro:
    """Clase para modelar un perro y su rutina de paseos."""

    def __init__(self, nombre, raza, edad):
        self.nombre = nombre
        self.raza = raza
        self.edad = edad
        self.horas_out = set()
        print(f'Hola, soy {nombre}, un {raza} de {edad} año(s)')

    # ── métodos ──────────────────────────────────────────────────────────────

    def ladrar(self):
        """Ladrido de 10 iteraciones alternando intensidad."""
        for i in range(10):
            print('Guau Guau!' if i % 2 == 0 else 'Guau Guau Guau!')

    def registrar_paseos(self, *horas):
        """
        Registra las horas de paseo diarias.
        
        Args:
            *horas: strings con formato 'HH:MM:SS', p.ej. '09:00:00'
        """
        for h in horas:
            self.horas_out.add(datetime.strptime(h, '%H:%M:%S').time())
        return self.horas_out

    def tiempo_hasta_paseo(self):
        """Devuelve cuánto falta para el próximo paseo."""
        now = datetime.now()
        proximos = []
        for hora in self.horas_out:
            dt = datetime.combine(datetime.today().date(), hora)
            if dt > now:
                proximos.append(dt - now)
        if proximos:
            return f'Faltan {str(min(proximos)).split(".")[0]} para sacar a {self.nombre}'
        return f'No hay más paseos programados hoy para {self.nombre}'

In [ ]:
cop = Perro('Cop', 'Chucho', 3)

In [ ]:
cop.ladrar()

In [ ]:
cop.registrar_paseos('09:00:00', '13:00:00', '15:30:00', '21:30:00', '23:30:00')

In [ ]:
# Prueba en distintas horas del día para ver qué devuelve
print(cop.tiempo_hasta_paseo())

### Listas vs Sets en los atributos

Nota que usamos un `set` para `horas_out` en lugar de una lista.  
Prueba a cambiar `set()` por `[]` y `add()` por `append()` para ver la diferencia:
- Con `list`: puede haber horas repetidas
- Con `set`: no hay duplicados, y el orden no está garantizado (pero da igual para nuestro uso)

---
## 4. Herencia

La **herencia** permite crear una clase nueva (clase hija) a partir de una clase existente
(clase padre), reutilizando y extendiendo su comportamiento.

```
Animal (padre / superclase)
  └── Perro (hijo / subclase)
```

`super().__init__(...)` llama al constructor del padre desde el hijo.

In [ ]:
class Animal:
    """Clase base para todos los animales."""

    def __init__(self, color):
        self.color = color

    def moverse(self):
        print('Me estoy moviendo... ¡a ver si son pulgas!')

    def dormir(self):
        print('Zzz... qué bien se duerme.')

In [ ]:
from datetime import datetime

class Perro(Animal):          # <-- Perro hereda de Animal
    """Subclase de Animal que añade comportamiento específico del perro."""

    def __init__(self, color, nombre, raza, edad):
        super().__init__(color)   # inicializar el padre primero
        self.nombre = nombre
        self.raza = raza
        self.edad = edad
        self.horas_out = set()
        print(f'Hola, soy {nombre}, un {raza} de {edad} año(s)')

    def ladrar(self):
        for i in range(6):
            print('Guau Guau!' if i % 2 == 0 else 'Guau Guau Guau!')

    def registrar_paseos(self, *horas):
        for h in horas:
            self.horas_out.add(datetime.strptime(h, '%H:%M:%S').time())
        return self.horas_out

In [ ]:
miperro = Perro('marrón', 'Cop', 'Chucho', 3)

In [ ]:
# Métodos heredados de Animal
miperro.moverse()
miperro.dormir()

In [ ]:
# Método propio de Perro
miperro.ladrar()

In [ ]:
# Atributos de ambas clases
print(miperro.color)    # de Animal
print(miperro.nombre)   # de Perro
print(isinstance(miperro, Perro))
print(isinstance(miperro, Animal))   # True! Perro IS-A Animal

---
## 5. Métodos especiales: `__repr__` y `__str__`

Los **dunder methods** (double underscore) son métodos mágicos que Python llama internamente.

- `__str__`: versión legible para humanos del objeto (la que usa `print()`)
- `__repr__`: versión técnica/inequívoca (para depuración y REPL)

Siempre es buena práctica definirlos en nuestras clases.

In [ ]:
class Perro:
    def __init__(self, nombre, raza, edad):
        self.nombre = nombre
        self.raza = raza
        self.edad = edad

    def __str__(self):
        return f'{self.nombre} ({self.raza}, {self.edad} años)'

    def __repr__(self):
        return f"Perro(nombre='{self.nombre}', raza='{self.raza}', edad={self.edad})"

cop = Perro('Cop', 'Chucho', 3)
print(cop)      # llama a __str__
cop             # en el REPL/notebook llama a __repr__

---
## 6. Métodos de clase y estáticos

### `@classmethod`

Un método de clase recibe la **clase** (`cls`) en lugar de la instancia (`self`).
Se usa habitualmente para crear **constructores alternativos** o trabajar con
atributos de clase (compartidos por todas las instancias).

### `@staticmethod`

Un método estático no recibe ni `self` ni `cls`. Es simplemente una función
que lógicamente pertenece a la clase pero no necesita acceder a ella.
Es la menos usada de las tres, pero útil para utilidades relacionadas con la clase.

> **Consejo práctico:** Si no tienes claro cuándo usar cada tipo, empieza con métodos
> normales. Los decoradores `@classmethod` y `@staticmethod` los usarás cuando
> los necesites específicamente.

In [ ]:
class Perro:
    # Atributo de clase — compartido por todas las instancias
    especie = 'Canis lupus familiaris'
    _registro = []              # lista de todos los perros creados

    def __init__(self, nombre, raza, edad):
        self.nombre = nombre
        self.raza = raza
        self.edad = edad
        Perro._registro.append(self)   # registrar al crearse

    def __repr__(self):
        return f"Perro('{self.nombre}', '{self.raza}', {self.edad})"

    # ── @classmethod: recibe cls, no self ─────────────────────────────────────
    @classmethod
    def cuantos_hay(cls):
        """Devuelve el número total de perros creados."""
        return f'Hay {len(cls._registro)} perro(s) registrado(s)'

    @classmethod
    def desde_string(cls, texto):
        """
        Constructor alternativo: crea un Perro a partir de un string
        con el formato 'nombre,raza,edad'.
        """
        nombre, raza, edad = texto.split(',')
        return cls(nombre.strip(), raza.strip(), int(edad.strip()))

    # ── @staticmethod: ni self ni cls ─────────────────────────────────────────
    @staticmethod
    def es_nombre_valido(nombre):
        """Valida que el nombre no esté vacío y no tenga números."""
        return bool(nombre) and nombre.replace(' ', '').isalpha()

In [ ]:
p1 = Perro('Cop', 'Chucho', 3)
p2 = Perro('Fury', 'Pastor Alemán', 5)

# Método de clase — se llama sobre la clase, no sobre una instancia
print(Perro.cuantos_hay())

# Constructor alternativo
p3 = Perro.desde_string('Tobe, Beagle, 2')
print(p3)

# Método estático
print(Perro.es_nombre_valido('Cop'))
print(Perro.es_nombre_valido('Cop123'))
print(Perro.es_nombre_valido(''))

---
## 7. `@property`: atributos calculados y con control de acceso

`@property` convierte un método en un **atributo de solo lectura** (o con setter
y deleter opcionales). Es útil para:

- Calcular un valor a partir de otros atributos sin almacenarlo
- Validar datos al asignar un valor

In [ ]:
class Perro:
    def __init__(self, nombre, raza, anio_nacimiento):
        self.nombre = nombre
        self.raza = raza
        self.anio_nacimiento = anio_nacimiento

    @property
    def edad(self):
        """Edad calculada dinámicamente a partir del año de nacimiento."""
        from datetime import date
        return date.today().year - self.anio_nacimiento

    @property
    def descripcion(self):
        return f'{self.nombre} — {self.raza}, {self.edad} años'

cop = Perro('Cop', 'Chucho', 2021)
print(cop.edad)         # lo usamos como atributo, aunque es un método
print(cop.descripcion)  # calculado al vuelo

In [ ]:
# Un property con setter: validar que la edad no sea negativa
class Perro:
    def __init__(self, nombre, raza, edad):
        self.nombre = nombre
        self.raza = raza
        self.edad = edad    # llama al setter

    @property
    def edad(self):
        return self._edad

    @edad.setter
    def edad(self, valor):
        if not isinstance(valor, int) or valor < 0:
            raise ValueError(f'La edad debe ser un entero no negativo, no {valor!r}')
        self._edad = valor

cop = Perro('Cop', 'Chucho', 3)
print(cop.edad)

In [ ]:
# Esto debe lanzar un ValueError
try:
    cop.edad = -1
except ValueError as e:
    print(f'Error: {e}')

---
## 8. Un ejemplo geoespacial: clase `PuntoGPS`

Para cerrar, construimos una clase que modela un punto GPS con todo lo aprendido.

In [ ]:
import math
from datetime import datetime

class PuntoGPS:
    """
    Punto geográfico con latitud, longitud y timestamp opcional.
    
    Usa WGS84 (el sistema de referencia del GPS estándar).
    """

    _contador = 0   # atributo de clase

    def __init__(self, lat: float, lon: float, nombre: str = '',
                 timestamp: datetime = None):
        if not (-90 <= lat <= 90):
            raise ValueError(f'Latitud fuera de rango: {lat}')
        if not (-180 <= lon <= 180):
            raise ValueError(f'Longitud fuera de rango: {lon}')
        self.lat = lat
        self.lon = lon
        self.nombre = nombre
        self.timestamp = timestamp or datetime.now()
        PuntoGPS._contador += 1

    # ── Dunders ───────────────────────────────────────────────────────────────

    def __repr__(self):
        return f'PuntoGPS({self.lat}, {self.lon}, nombre={self.nombre!r})'

    def __str__(self):
        ns = 'N' if self.lat >= 0 else 'S'
        ew = 'E' if self.lon >= 0 else 'W'
        return f'{abs(self.lat):.4f}°{ns}, {abs(self.lon):.4f}°{ew}'

    def __eq__(self, other):
        return isinstance(other, PuntoGPS) and self.lat == other.lat and self.lon == other.lon

    # ── Property ──────────────────────────────────────────────────────────────

    @property
    def coordenadas(self):
        return (self.lat, self.lon)

    # ── Classmethod ───────────────────────────────────────────────────────────

    @classmethod
    def desde_dms(cls, lat_d, lat_m, lat_s, lat_hem,
                  lon_d, lon_m, lon_s, lon_hem, nombre=''):
        """Crea un punto desde grados-minutos-segundos."""
        lat = (lat_d + lat_m/60 + lat_s/3600) * (1 if lat_hem == 'N' else -1)
        lon = (lon_d + lon_m/60 + lon_s/3600) * (1 if lon_hem == 'E' else -1)
        return cls(lat, lon, nombre)

    @classmethod
    def total_creados(cls):
        return cls._contador

    # ── Staticmethod ──────────────────────────────────────────────────────────

    @staticmethod
    def haversine(p1, p2):
        """Distancia en km entre dos PuntoGPS usando la fórmula de Haversine."""
        R = 6371
        lat1, lon1 = math.radians(p1.lat), math.radians(p1.lon)
        lat2, lon2 = math.radians(p2.lat), math.radians(p2.lon)
        dlat = lat2 - lat1
        dlon = lon2 - lon1
        a = math.sin(dlat/2)**2 + math.cos(lat1)*math.cos(lat2)*math.sin(dlon/2)**2
        return R * 2 * math.asin(math.sqrt(a))

In [ ]:
# Doñana (EBD-CSIC)
donana   = PuntoGPS(37.0, -6.4, 'Doñana')
# Sevilla
sevilla  = PuntoGPS(37.3886, -5.9823, 'Sevilla')
# Marismas del Odiel
odiel    = PuntoGPS(37.2, -6.9, 'Odiel')

print(donana)
print(repr(donana))
print('Coordenadas:', donana.coordenadas)

In [ ]:
dist = PuntoGPS.haversine(donana, sevilla)
print(f'Distancia Doñana – Sevilla: {dist:.1f} km')

In [ ]:
# Constructor alternativo desde grados-minutos-segundos
donana_dms = PuntoGPS.desde_dms(37, 0, 0, 'N', 6, 24, 0, 'W', 'Doñana DMS')
print(donana_dms)

In [ ]:
print('Total puntos creados:', PuntoGPS.total_creados())

---
## Resumen

| Concepto | Para qué sirve |
|----------|---------------|
| `class` + `__init__` | Definir la plantilla y el constructor |
| `self` | Referencia a la instancia actual |
| Herencia | Reutilizar y extender comportamiento |
| `super()` | Llamar al constructor/método del padre |
| `__str__` / `__repr__` | Representación del objeto como string |
| `@classmethod` | Constructores alternativos, atributos de clase |
| `@staticmethod` | Funciones utilitarias relacionadas con la clase |
| `@property` | Atributos calculados o con validación |

> **Siguiente paso:** En el Día 2 trabajaremos con Shapely, Fiona y GeoPandas.
> Todos estos paquetes son colecciones de clases, y ahora entendemos cómo funcionan por dentro.

---
*GeoPython 2026 — CSIC / Estación Biológica de Doñana*